# Tutorial 1: Dataset Registration & Schema Generation

### Demonstrating how to register datasets on the Rhino Health Federated Computing Platform (FCP) and auto-generate Data Schemas using the Rhino Health Python SDK.

---

## What Is Happening Here?

**Registering a dataset does not upload any data.** It tells the Rhino client node:
> *"There is a file at this path. Confirm it is accessible and record its structure."*

The Rhino agent reads the file **locally on the client node**, collects metadata (row count, column names, inferred types), and sends only that metadata to the Rhino cloud. The underlying data never leaves the client.

**A Data Schema** describes the structure of a dataset — its column names and their data types. Schemas are used to:
- Validate that incoming data matches expectations before a run
- Enable federated analytics and data harmonization (Tutorials 3 and 4)
- Enforce type safety when running Code Objects on your data

> **Can I do this in the UI instead?**  
> Yes — every step in this notebook can also be performed manually in the FCP Dashboard under **Projects → Datasets → Import Dataset**. This notebook automates those same operations via the SDK.

---

## Prerequisites
1. **Tutorial 0 complete** — the 3 CSV source files (`patients.csv`, `encounters.csv`, `procedures.csv`) are accessible on the Rhino client node
2. **Active FCP credentials** — your Rhino Health username (email) and password
3. **`PROJECT_UID`** — find this in the FCP Dashboard: log in → from the Projects landing page, click the three-dot menu (⋮) on your project tile → **Copy UID**

## What This Notebook Produces
- 3 registered Datasets: Patients, Encounters, Procedures
- 3 Data Schemas (one per dataset, with corrected field types where needed)
- 6 UIDs to carry forward into Tutorial 2

## 0. Load All Necessary Libraries

### Ensure that you are running this notebook in the correct kernel.
### If needed, install the required libraries by uncommenting the following line.

In [ ]:
from getpass import getpass
import rhino_health as rh
from rhino_health import ApiEnvironment
import rhino_health.lib.endpoints.dataset.dataset_dataclass
import rhino_health.lib.endpoints.data_schema.data_schema_dataclass
from rhino_health.lib.endpoints.dataset.dataset_dataclass import DatasetCreateInput
from rhino_health.lib.endpoints.data_schema.data_schema_dataclass import DataSchemaCreateInput

## 1. Log In to the Rhino Health Platform

#### Replace the values for the following variables before running this cell:
1. `my_username` — your Rhino Health email address
2. `PROJECT_UID` — the UID of the project you created in the FCP Dashboard (see Prerequisites above)

A password prompt will appear when you run the cell. Type your password and press **Enter**.  
Your password is never stored in the notebook.

In [ ]:
my_username = "<ENTER_YOUR_EMAIL_HERE>"         # Replace with your username (email)
PROJECT_UID = "<ENTER_YOUR_PROJECT_UID_HERE>"   # Replace with your Project UID
# When creating your project, it is highly recommended to set the k-anonymization parameter to 1 instead of the default value of 5 for testing purposes, as this will allow you to see all records in the dataset without suppression. However, for production use, it is recommended to set k to a value of at least 5 to ensure patient privacy.

print("Logging in...")
session = rh.login(username=my_username, password=getpass(), rhino_api_url=ApiEnvironment.PROD_AWS_URL) # e.g., PROD_AWS_URL, STAGING_AWS_URL, DEV2_AWS_URL, SOLUTIONS_GCP_URL
print(f"Logged in successfully as <{my_username}>.")

In [ ]:
project   = session.project.get_projects(project_uids=[PROJECT_UID])[0]
workgroup = session.project.get_collaborating_workgroups(PROJECT_UID)[0]

print(f"Project \t({project.uid}): \t{project.name}")
print(f"Workgroup \t({workgroup.uid}): \t{workgroup.name}")

## 2. Define File Paths

The paths below tell the Rhino agent where to find each CSV file **on the client node's local filesystem**.

### Two Ways to Provide Data

**Option A — Client node filesystem (default)**  
Files have been copied directly onto the Rhino client VM (e.g., via Tutorial 0's data transfer step).  
Use paths under `/rhino_data/` as shown below.

**Option B — Client-mounted storage (S3, GCS, Azure Blob)**  
If your Rhino client is configured to mount an external storage bucket, files in that bucket are accessible under `/rhino_data/external/<provider>/<path-within-bucket>` — no separate copy step needed. Adjust the paths to match your bucket structure.

> **Not sure which option applies to you?** Check with your Rhino Health administrator, or look for a `/rhino_data/external/` directory on the client node.

In [ ]:
# Option A: Client-mounted external storage (e.g., S3 bucket mounted to the client)
# Uncomment and adjust these paths if your data lives in a mounted storage bucket.
# The provider prefix ("s3", "gcs", "azure") may vary depending on your environment and client configuration.
#
PATIENTS_PATH   = "/rhino_data/external/s3/intro_to_data_engineering/patients.csv"
ENCOUNTERS_PATH = "/rhino_data/external/s3/intro_to_data_engineering/encounters.csv"
PROCEDURES_PATH = "/rhino_data/external/s3/intro_to_data_engineering/procedures.csv"

# Option B: Files copied directly to the client node filesystem
# Replace with the actual paths where your files are located on the client.
# PATIENTS_PATH   = "/rhino_data/intro_to_data_engineering/patients.csv"
# ENCOUNTERS_PATH = "/rhino_data/intro_to_data_engineering/encounters.csv"
# PROCEDURES_PATH = "/rhino_data/intro_to_data_engineering/procedures.csv"

print("File paths configured:")
print(f"  Patients:   {PATIENTS_PATH}")
print(f"  Encounters: {ENCOUNTERS_PATH}")
print(f"  Procedures: {PROCEDURES_PATH}")

---
## 3. Dataset 1 of 3 — Patients

**Source columns:** `patientID`, `YearOfBirth`, `Gender`, `Race`, `Ethnicity`

**Known data quality issues in this example file** (to be discovered in Tutorial 2 and addressed in Tutorial 3):
- `Gender` has casing inconsistencies: `"female"`, `"FEMALE"`, `"Male"`, etc.
- Two rows have null `Gender` values
- Row 41: `YearOfBirth = 10000` (invalid — out-of-range year)
- Row 45: `Gender = "Alien"` (invalid enum value)
- Last row is a duplicate of row 3

**What to watch for in the schema:** `YearOfBirth` is sometimes inferred as `float` instead of `integer` due to null or malformed values in the source data. See step 3.3 below.

### 3.1 Register the Patients Dataset

In [ ]:
patients_dataset = session.dataset.add_dataset(DatasetCreateInput(
    name="Patients — Site A",
    description="Patient demographic data for Tutorial 1 — Site A",
    project_uid=PROJECT_UID,
    workgroup_uid=workgroup.uid,
    csv_filesystem_location=PATIENTS_PATH,
    data_schema_uid=None,       # This will auto-generate a schema based on the CSV header row. For custom schemas, create a DataSchema and provide its UID here.
    is_data_deidentified=True,
    method="filesystem",
))

PATIENTS_DATASET_UID = patients_dataset.uid
PATIENTS_DATASET_NAME = patients_dataset.name
print(f"Patients dataset registered.")
print(f"  Name: {PATIENTS_DATASET_NAME}")
print(f"  UID:  {PATIENTS_DATASET_UID}")
print(f"  Rows: {patients_dataset.num_cases or 'pending — refresh in 1 to 2 min'}")

### 3.2 Check Auto-Generated Schema for Patients

Since the schema was auto-generated, it will be automatically named `<DATASET_NAME> schema`

There are multiple ways to check:
* Option A: run the SDK code below to pull the schema and print the fields
* Option B: In the UI, go to the `Datasets` tab, and click on the Schema attached to the dataset
* Option C: In the UI, go to the `Data Schemas` tab and click on the Schema directly

In [ ]:
patients_schema = patients_dataset.data_schema
PATIENTS_SCHEMA_NAME = patients_schema.name
print(f"Schema Name: {PATIENTS_SCHEMA_NAME}")
PATIENTS_SCHEMA_UID = patients_schema.uid
print(f"Schema UID: {PATIENTS_SCHEMA_UID}")
print("─" * 65)

fields = patients_schema.schema_fields.root
num_cols = len(patients_schema.schema_fields.root)
print(f"Number of Columns: {num_cols}")
print("─" * 65)

names = patients_schema.schema_fields.field_names
for field in patients_schema.schema_fields.root:
    print(f"{field.name}: {field.type}")


### 3.3 Correct the Schema (if needed) - YearOfBirth -> Integer

If `YearOfBirth` was inferred as `float` instead of `integer`, run the correction block below.

This can happen when the CSV contains null or malformed values in a column — the Rhino agent defaults to `float` when it cannot confidently determine the type.

> **Tip:** You can also correct schemas directly in the FCP Dashboard:  
Dashboard → Data Schemas → Click the Schema → Edit Schema

> **Note:** Editing a schema will always create a NEW version


In [ ]:
# Run this if YearOfBirth was incorrectly inferred as float to convert it to integer:

corrected_fields = []
for field in patients_schema.schema_fields.root:
    if field.name == "YearOfBirth":
        field.type = "Integer"
        print(f"Corrected '{field.name}': float → integer\n")
    corrected_fields.append(field)

col_names = [f.name for f in corrected_fields]
col_types  = [f.type or "string" for f in corrected_fields]

schema_rows = [
    "Variable Name,"          + ",".join(col_names)                       + "\n",
    "Identifier,"             + ",".join([""] * len(col_names))           + "\n",
    "Description,"            + ",".join([""] * len(col_names))           + "\n",
    "Type,"                   + ",".join(col_types)                       + "\n",
    "Type Parameters,"        + ",".join([""] * len(col_names))           + "\n",
    "Units,"                  + ",".join([""] * len(col_names))           + "\n",
    "Contains Sensitive Data?," + ",".join(["False"] * len(col_names))   + "\n",
    "Permissions,"            + ",".join(["default"] * len(col_names))    + "\n",
]

patients_schema = session.data_schema.create_data_schema(
    DataSchemaCreateInput(
        name=patients_schema.name,
        description=patients_schema.description or "",
        project_uid=PROJECT_UID,
        primary_workgroup_uid=workgroup.uid,
        schema_fields=schema_rows,
    ),
    return_existing=False,
    add_version_if_exists=True,
)

PATIENTS_SCHEMA_UID = patients_schema.uid
print(f"Schema updated (v{patients_schema.version}): uid={PATIENTS_SCHEMA_UID}")
for field in patients_schema.schema_fields.root:
    print(f"  {field.name}: {field.type}")

### 3.4 Update Patients Dataset (if you ran 3.3)

If you ran step 3.3, then you need to run this step to update the dataset to work off the newly revised schema. Otherwise, you can skip this step.

In [ ]:
patients_dataset = session.dataset.add_dataset(
    DatasetCreateInput(
        name=patients_dataset.name,
        description=patients_dataset.description or "",
        project_uid=PROJECT_UID,
        workgroup_uid=workgroup.uid,
        csv_filesystem_location=PATIENTS_PATH,     # same CSV as before
        data_schema_uid=PATIENTS_SCHEMA_UID,       # corrected schema
        method="filesystem",
        is_data_deidentified=True,
    ),
    return_existing=False,
    add_version_if_exists=True,                    # creates new version of the dataset
)

PATIENTS_DATASET_UID = patients_dataset.uid
print(f"Dataset updated (v{patients_dataset.version}): uid={PATIENTS_DATASET_UID}")
print(f"Schema UID: {patients_dataset.data_schema_uid}")

---
## 4. Dataset 2 of 3 — Encounters

**Source columns:** `patientID`, `visitID`, `DateOfService`, `TypeOfService`

**Known data quality issues in this example file:**
- `TypeOfService` has casing inconsistencies: `"outpatient"`, `"INPATIENT"`, `"Outpatient"`, etc.
- Row 73: `DateOfService` is null/empty
- Last row is a duplicate of row 3

**What to watch for in the schema:** `DateOfService` is **commonly inferred as `string`** rather than `date`.  
The correction in step 4.3 is **required** before proceeding to Tutorial 3 — date type is needed for OMOP harmonization.

### 4.1 Register the Encounters Dataset

In [ ]:
encounters_dataset = session.dataset.add_dataset(DatasetCreateInput(
    name="Encounters — Site A",
    description="Patient encounter / visit data for Tutorial 1 — Site A",
    project_uid=PROJECT_UID,
    workgroup_uid=workgroup.uid,
    csv_filesystem_location=ENCOUNTERS_PATH,
    data_schema_uid=None,
    is_data_deidentified=True,
    method="filesystem",
))

ENCOUNTERS_DATASET_UID = encounters_dataset.uid
print(f"Encounters dataset registered.")
print(f"  Name: {encounters_dataset.name}")
print(f"  UID:  {ENCOUNTERS_DATASET_UID}")
print(f"  Rows: {encounters_dataset.num_cases or 'pending — refresh in 1 to 2 min'}")

### 4.2 Check Auto-Generated Schema for Encounters

Since the schema was auto-generated, it will be automatically named `<DATASET_NAME> schema`

There are multiple ways to check:
* Option A: run the SDK code below to pull the schema and print the fields
* Option B: In the UI, go to the `Datasets` tab, and click on the Schema attached to the dataset
* Option C: In the UI, go to the `Data Schemas` tab and click on the Schema directly

In [ ]:
encounters_schema = encounters_dataset.data_schema
ENCOUNTERS_SCHEMA_NAME = encounters_schema.name
print(f"Schema Name: {ENCOUNTERS_SCHEMA_NAME}")
ENCOUNTERS_SCHEMA_UID = encounters_schema.uid
print(f"Schema UID: {ENCOUNTERS_SCHEMA_UID}")
print("─" * 65)

fields = encounters_schema.schema_fields.root
num_cols = len(encounters_schema.schema_fields.root)
print(f"Number of Columns: {num_cols}")
print("─" * 65) 

names = encounters_schema.schema_fields.field_names
for field in encounters_schema.schema_fields.root:
    print(f"{field.name}: {field.type}")

### 4.3 Correct the Schema (if needed) — DateOfService → Date

If `DateOfService` was inferred as `string`, this correction is **required** before proceeding to Tutorial 3.  

The OMOP harmonization engine relies on date-typed columns to correctly map temporal fields.

In [ ]:
# Run this if DateOfService was incorrectly inferred as string to convert it to date:

corrected_fields = []
for field in encounters_schema.schema_fields.root:
    if field.name == "DateOfService":
        field.type = "Date"
        print(f"  Corrected '{field.name}': String → Date")
    corrected_fields.append(field)

col_names = [f.name for f in corrected_fields]
col_types  = [f.type or "String" for f in corrected_fields]

schema_rows = [
    "Variable Name,"            + ",".join(col_names)                     + "\n",
    "Identifier,"               + ",".join([""] * len(col_names))         + "\n",
    "Description,"              + ",".join([""] * len(col_names))         + "\n",
    "Type,"                     + ",".join(col_types)                     + "\n",
    "Type Parameters,"          + ",".join([""] * len(col_names))         + "\n",
    "Units,"                    + ",".join([""] * len(col_names))         + "\n",
    "Contains Sensitive Data?," + ",".join(["False"] * len(col_names))   + "\n",
    "Permissions,"              + ",".join(["default"] * len(col_names))  + "\n",
]

encounters_schema = session.data_schema.create_data_schema(
    DataSchemaCreateInput(
        name=encounters_schema.name,
        description=encounters_schema.description or "",
        project_uid=PROJECT_UID,
        primary_workgroup_uid=workgroup.uid,
        schema_fields=schema_rows,
    ),
    return_existing=False,
    add_version_if_exists=True,
)

ENCOUNTERS_SCHEMA_UID = encounters_schema.uid
print(f"Encounters schema updated (v{encounters_schema.version}): uid={ENCOUNTERS_SCHEMA_UID}")
for field in encounters_schema.schema_fields.root:
    print(f"  {field.name}: {field.type}")

### 4.4 Attach the Schema to the Encounters Dataset (if you ran 4.3)

If you ran step 4.3, then you need to run this step to update the dataset to work off the newly revised schema. Otherwise, you can skip this step.

In [ ]:
encounters_dataset = session.dataset.add_dataset(
    DatasetCreateInput(
        name=encounters_dataset.name,
        description=encounters_dataset.description or "",
        project_uid=PROJECT_UID,
        workgroup_uid=workgroup.uid,
        csv_filesystem_location=ENCOUNTERS_PATH,
        data_schema_uid=ENCOUNTERS_SCHEMA_UID,
        method="filesystem",
        is_data_deidentified=True,
    ),
    return_existing=False,
    add_version_if_exists=True,
)

ENCOUNTERS_DATASET_UID = encounters_dataset.uid
print(f"Dataset updated (v{encounters_dataset.version}): uid={ENCOUNTERS_DATASET_UID}")
print(f"Schema UID: {encounters_dataset.data_schema_uid}")

---
## 5. Dataset 3 of 3 — Procedures

**Source columns:** `patientID`, `visitID`, `ProcedureDate`, `ProcedureDescription`, `ProcedureCode`, `ProcedureCategory`

**Known data quality issues in this example file:**
- Row 79: `ProcedureDescription` is null (acceptable — description is not required for OMOP mapping)
- Row 92: `ProcedureCode` is null (this row will be dropped in Tutorial 3's cleaning step)
- Last row is a duplicate of row 3

**What to watch for in the schema:**
- `ProcedureDate` is commonly inferred as `string` — correction **required** (same reason as `DateOfService` above)
- `ProcedureCode` may be inferred as `float` if the null in row 92 causes the agent to widen the type

### 5.1 Register the Procedures Dataset

In [ ]:
procedures_dataset = session.dataset.add_dataset(DatasetCreateInput(
    name="Procedures — Site A",
    description="Clinical procedure records for Tutorial 1 — Site A",
    project_uid=PROJECT_UID,
    workgroup_uid=workgroup.uid,
    csv_filesystem_location=PROCEDURES_PATH,
    data_schema_uid=None,
    is_data_deidentified=True,
    method="filesystem",
))

PROCEDURES_DATASET_UID = procedures_dataset.uid
print(f"Procedures dataset registered.")
print(f"  Name: {procedures_dataset.name}")
print(f"  UID:  {PROCEDURES_DATASET_UID}")
print(f"  Rows: {procedures_dataset.num_cases or 'pending — refresh in 1 to 2 min'}")

### 5.2 Check Auto-Generated Schema for Procedures

Since the schema was auto-generated, it will be automatically named `<DATASET_NAME> schema`

There are multiple ways to check:
* Option A: run the SDK code below to pull the schema and print the fields
* Option B: In the UI, go to the `Datasets` tab, and click on the Schema attached to the dataset
* Option C: In the UI, go to the `Data Schemas` tab and click on the Schema directly

In [ ]:
procedures_schema = procedures_dataset.data_schema
PROCEDURES_SCHEMA_NAME = procedures_schema.name
print(f"Schema Name: {PROCEDURES_SCHEMA_NAME}")
PROCEDURES_SCHEMA_UID = procedures_schema.uid
print(f"Schema UID: {PROCEDURES_SCHEMA_UID}")
print("─" * 65)

fields = procedures_schema.schema_fields.root
num_cols = len(procedures_schema.schema_fields.root)
print(f"Number of Columns: {num_cols}")
print("─" * 65)

names = procedures_schema.schema_fields.field_names
for field in procedures_schema.schema_fields.root:
    print(f"{field.name}: {field.type}")

### 5.3 Correct the Schema (if needed) — ProcedureDate -> Date & ProcedureCode -> Integer

In [ ]:
# Run this if ProcedureDate was incorrectly inferred as String to convert it to Date, and if ProcedureCode was incorrectly inferred as Float to convert it to Integer:

corrected_fields = []
for field in procedures_schema.schema_fields.root:
    if field.name == "ProcedureDate":
        field.type = "Date"
        print(f"  Corrected '{field.name}': String → Date")
    if field.name == "ProcedureCode" and field.type == "Float":
        field.type = "Integer"
        print(f"  Corrected '{field.name}': Float → Integer")
    corrected_fields.append(field)

col_names = [f.name for f in corrected_fields]
col_types  = [f.type or "String" for f in corrected_fields]

schema_rows = [
    "Variable Name,"            + ",".join(col_names)                     + "\n",
    "Identifier,"               + ",".join([""] * len(col_names))         + "\n",
    "Description,"              + ",".join([""] * len(col_names))         + "\n",
    "Type,"                     + ",".join(col_types)                     + "\n",
    "Type Parameters,"          + ",".join([""] * len(col_names))         + "\n",
    "Units,"                    + ",".join([""] * len(col_names))         + "\n",
    "Contains Sensitive Data?," + ",".join(["False"] * len(col_names))   + "\n",
    "Permissions,"              + ",".join(["default"] * len(col_names))  + "\n",
]

procedures_schema = session.data_schema.create_data_schema(
    DataSchemaCreateInput(
        name=procedures_schema.name,
        description=procedures_schema.description or "",
        project_uid=PROJECT_UID,
        primary_workgroup_uid=workgroup.uid,
        schema_fields=schema_rows,
    ),
    return_existing=False,
    add_version_if_exists=True,
)

PROCEDURES_SCHEMA_UID = procedures_schema.uid
print(f"Schema updated (v{procedures_schema.version}): uid={PROCEDURES_SCHEMA_UID}")
for field in procedures_schema.schema_fields.root:
    print(f"  {field.name}: {field.type}")

### 5.4 Update Procedures Dataset (if you ran 5.3)

If you ran step 5.3, then you need to run this step to update the dataset to work off the newly revised schema. Otherwise, you can skip this step.

In [ ]:
procedures_dataset = session.dataset.add_dataset(
    DatasetCreateInput(
        name=procedures_dataset.name,
        description=procedures_dataset.description or "",
        project_uid=PROJECT_UID,
        workgroup_uid=workgroup.uid,
        csv_filesystem_location=PROCEDURES_PATH,
        data_schema_uid=PROCEDURES_SCHEMA_UID,
        method="filesystem",
        is_data_deidentified=True,
    ),
    return_existing=False,
    add_version_if_exists=True,
)

PROCEDURES_DATASET_UID = procedures_dataset.uid
print(f"Dataset updated (v{procedures_dataset.version}): uid={PROCEDURES_DATASET_UID}")
print(f"Schema UID: {procedures_dataset.data_schema_uid}")

---
## 6. Double-Check in the FCP Dashboard

After running all cells above, verify the following in the FCP Dashboard:

### 6.1 — Datasets Registered
Navigate to: **Projects (Landing Page) → [Your Project] → Datasets tab**
- All 3 datasets should appear: 
    * `Patients — Site A`
    * `Encounters — Site A`
    * `Procedures — Site A`
- Each should show the correct **Number of Rows** (100) & have an associated data schema
- If `Number of Rows` shows `—` or is missing, wait 1–2 minutes and refresh (row counts are computed asynchronously)
- If you ran steps (X.3) & (X.4), you will see multiple versions of each corresponding dataset

### 6.2 — Schemas Generated
Navigate to: **Projects (Landing Page) → [Your Project] → Data Schemas tab**
- 3 schemas should appear (one per dataset):
    * `Patients — Site A schema`
    * `Encounters — Site A schema`
    * `Procedures — Site A schema`
- Click any schema to inspect the field list and verify all types look correct
- If a type is wrong, you can edit it directly here (will create a new version), or re-run the corresponding correction cell (step X.3) for the dataset
- If you ran steps (X.3) & (X.4), you will see multiple versions of each corresponding schema

### 6.3 — Schemas Attached to Datasets
Navigate to: **Projects (Landing Page) → [Your Project] → Datasets tab**
- The **Schema** field on the dataset detail page should show the correct schema name, not be blank
- If it is blank, or the schema is wrong, re-run the corresponding `update_dataset` cell (step X.4) for that dataset and assign it to the appropriate schema

### 6.4 — What to Do If Something Looks Wrong
| Symptom | Fix |
|---|---|
| Unable to generate schemas | You must have an associated, online client, even if you are not uploading data |
| Dataset not appearing | Check that the CSV path is correct, the file is accessible on the client, and the client is online - (if using client-mounted storage, ensure it is connected) |
| Row count is 0 | The file may be empty or unreadable — verify permissions on the client |
| Wrong field type in schema | Run the corresponding correction cell for the dataset (Step X.3), or edit directly via the UI |
| Schema not attached | Run the corresponding correction cell for that dataset (Step X.4)|

---
## 7. Summary — Copy These UIDs for the Next Tutorial

Run the cell below to print all 6 UIDs you will need in Tutorial 2.  
Copy them into the configuration section at the top of the next notebook.

In [ ]:
print("=" * 68)
print("  Tutorial 1 Complete — save these UIDs for Tutorial 2")
print("=" * 68)
print()
print("# Dataset UIDs")
print(f"PATIENTS_DATASET_UID   = '{PATIENTS_DATASET_UID}'")
print(f"ENCOUNTERS_DATASET_UID = '{ENCOUNTERS_DATASET_UID}'")
print(f"PROCEDURES_DATASET_UID = '{PROCEDURES_DATASET_UID}'")
print()
print("# Schema UIDs")
print(f"PATIENTS_SCHEMA_UID    = '{PATIENTS_SCHEMA_UID}'")
print(f"ENCOUNTERS_SCHEMA_UID  = '{ENCOUNTERS_SCHEMA_UID}'")
print(f"PROCEDURES_SCHEMA_UID  = '{PROCEDURES_SCHEMA_UID}'")
print()
print("=" * 68)
print()
print("Continue to: Tutorial 2 — Data Discovery")